# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# metadata is an object; access attributes directly
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
# Optionally show some key metadata
print(f"Version: {dataset.metadata.version}")
print(f"Cite as: {dataset.metadata.citeAs}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Let's list all available record sets, their fields, and columns using their `@id` fields.

In [ ]:
# List all record sets, their fields, and columns using @id
record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No record sets found in the metadata.")
else:
    for rs in record_sets:
        print(f"RecordSet Name: {rs.name}")
        print(f"  @id: {rs.id}")
        print(f"  Description: {rs.description}")
        if hasattr(rs, 'fields') and rs.fields:
            print(f"  Fields:")
            for field in rs.fields:
                print(f"    Field: {field.name} | @id: {field.id}")
                if hasattr(field, 'columns') and field.columns:
                    for col in field.columns:
                        print(f"      Column: {col.name} | @id: {col.id}")
        print('-' * 60)

## 3. Data Extraction
Load data from one or more record sets into a DataFrame for analysis. Use the record set and field/column `@id`s from the overview.

If record sets appear empty above, consult the Croissant schema in a browser or with `dataset.metadata.record_sets` for valid IDs.

In [ ]:
# Example extraction using observed @id's
# If the dataset has no record_sets in metadata but exposes default data,
# we can try the first available one based on the schema documentation.

if not record_sets:
    # Try to get a record set id from the data files (common in datasets without explicit record sets)
    print("No explicit record sets found in metadata. Attempting dataset.records() with default parameters...")
    try:
        df = pd.DataFrame(list(dataset.records()))
        print("Loaded records. Columns:", df.columns.tolist())
        display(df.head())
        # Create a mapping similar to record_sets for later steps
        dataframes = {'default': df}
        main_rs_id = 'default'
    except Exception as e:
        print(f"Could not extract records: {e}")
        dataframes = {}
        main_rs_id = None
else:
    record_set_ids = [rs.id for rs in record_sets]
    print("Extracting RecordSet DataFrames for these @id's:", record_set_ids)
    dataframes = {}
    for record_set_id in record_set_ids:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded: {record_set_id} -> Columns: {dataframes[record_set_id].columns.tolist()}")
    main_rs_id = record_set_ids[0] if record_set_ids else None

# Display the first few rows for the main record set
if main_rs_id and main_rs_id in dataframes:
    print("First few rows of the main record set:")
    display(dataframes[main_rs_id].head())
else:
    print("Could not determine main record set.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section can include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.

In [ ]:
import numpy as np

if main_rs_id and main_rs_id in dataframes:
    df = dataframes[main_rs_id]
    print(f"Columns in main DataFrame: {df.columns.tolist()}")
    # Pick a numeric field by common name (from mass spec datasets, e.g., "MW" for molecular weight or similar)
    possible_numeric_fields = [
        col for col in df.columns 
        if any(x in col.lower() for x in ['mw', 'weight', 'mass', 'score', 'amount', 'abundance', 'count', 'coverage', 'intensity'])
           and (df[col].dtype in [np.float64, np.float32, np.int64, np.int32] or pd.api.types.is_numeric_dtype(df[col]))
    ]
    print(f"Detected numeric fields for analysis: {possible_numeric_fields}")
    numeric_field = possible_numeric_fields[0] if possible_numeric_fields else None
    if numeric_field:
        # Apply filtering (e.g., molecular weight > threshold)
        threshold = df[numeric_field].quantile(0.75) if not df[numeric_field].isna().all() else 0  # use top quartile if possible
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df[[numeric_field]].head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a categorical field if found
        possible_group_fields = [
            col for col in df.columns if col != numeric_field and df[col].dtype == 'object'
        ]
        group_field = possible_group_fields[0] if possible_group_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Average {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No main DataFrame available for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. For example, distribution of the selected numeric field, or relationship between two fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id and main_rs_id in dataframes and numeric_field:
    df = dataframes[main_rs_id]
    # Plot histogram of numeric_field
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    # If group_field is available, boxplot of numeric_field by group_field
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print('Visualization skipped: No numeric field found.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded the FAIR^2 dataset using `mlcroissant` and reviewed its metadata and available record sets.
- Data was extracted into DataFrames using provided Croissant schema `@id`s.
- Common exploratory data analysis steps and visualizations were applied to the protein data.
- You can extend this notebook to perform in-depth analysis, modeling, or export selected subsets for downstream bioinformatics tasks.